# Web3 Python 100本ノック：第1章
## §1-10 (Bコース)：オンチェーン・イベントウォッチャーを構築せよ

システムの裏側やデータ構造に興味があるエンジニア向けの最終課題です。

ブロックチェーンは、約12秒に1回のペースで世界中のトランザクションをまとめた「新しいブロック」を生成し続けています。
この問題では、無限ループを用いて最新のブロック生成を監視（ポーリング）し、新着ブロックを検知した瞬間にその詳細データ（トランザクション数やGas使用量）をターミナルにストリーミング出力する、**常駐型の監視デーモン**の基礎を構築します。

> **注意**: このセルは手動で停止（停止ボタンを押す、またはキーボードインタラプト）するまで無限に動き続けます。

In [ ]:
!pip install web3==7.16.0

## ブロックデータの解析と常駐化

取得したブロックデータは辞書型（Dict）で返ってきます。
このスクリプトは単体で動かすだけでなく、将来的にLinuxの `systemd` などでバックグラウンドサービス化したり、標準出力をパイプしてターミナルUI（TUI）のダッシュボードにリアルタイム描画するための強力なバックエンド（データソース）としてそのまま転用できます。

In [ ]:
from web3 import Web3
import time
from datetime import datetime

RPC_URL = "https://eth.drpc.org"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
w3 = Web3(Web3.HTTPProvider(RPC_URL, request_kwargs={'headers': headers, 'timeout': 5}))

print("オンチェーン・イベントウォッチャーを起動しました。")
print("新しいブロックの生成を監視しています... (停止するにはセルの実行を中断してください)\n")

# 前回確認したブロック番号を保持する変数
last_seen_block = w3.eth.block_number
print(f"[INIT] 現在の最新ブロック: {last_seen_block}")
print("=" * 60)

try:
    # デーモンとして無限ループで監視
    # ※ デモ用にループ回数を5回までに制限しています。永続化する場合は while True: に変更してください。
    for _ in range(5):
        # 最新のブロック番号を再取得
        current_block = w3.eth.block_number
        
        # 新しいブロックが追加されていた場合
        if current_block > last_seen_block:
            for block_num in range(last_seen_block + 1, current_block + 1):
                # ブロックの詳細データを取得（Trueにすると全トランザクションの詳細を含むため重くなります。今回はFalse）
                block_data = w3.eth.get_block(block_num, full_transactions=False)
                
                # タイムスタンプを人間が読める日時に変換
                block_time = datetime.fromtimestamp(block_data.timestamp).strftime('%Y-%m-%d %H:%M:%S')
                
                # トランザクションの数を計算
                tx_count = len(block_data.transactions)
                
                print(f" [NEW BLOCK] {block_num}")
                print(f"   タイムスタンプ : {block_time}")
                print(f"   ブロックハッシュ: {block_data.hash.hex()[:15]}...")
                print(f"   トランザクション: {tx_count} 件")
                print(f"   Gas使用量      : {block_data.gasUsed:,} / {block_data.gasLimit:,}")
                print("-" * 60)
                
            # 最後に確認したブロックを更新
            last_seen_block = current_block
            
        # イーサリアムのブロック生成間隔（約12秒）に合わせて待機
        # ※パブリックノードの負荷を避けるため、10秒程度のsleepを入れるのが鉄則です
        time.sleep(10)
        
except KeyboardInterrupt:
    print("\n 監視デーモンを安全に停止しました。")
except Exception as e:
    print(f"\n エラーにより監視を停止しました: {e}")
else:
    print("\n デモ実行（5ループ）が完了しました。")

オンチェーン・イベントウォッチャーを起動しました。
新しいブロックの生成を監視しています... (停止するにはセルの実行を中断してください)

[INIT] 現在の最新ブロック: 25664699
 [NEW BLOCK] 25664700
   タイムスタンプ : 2026-08-02 13:06:35
   ブロックハッシュ: 3f9a15a63d8018b...
   トランザクション: 178 件
   Gas使用量      : 21,876,373 / 60,000,000
------------------------------------------------------------
 [NEW BLOCK] 25664701
   タイムスタンプ : 2026-08-02 13:06:47
   ブロックハッシュ: b4964912cc4de55...
   トランザクション: 112 件
   Gas使用量      : 9,124,847 / 60,000,000
------------------------------------------------------------
 [NEW BLOCK] 25664702
   タイムスタンプ : 2026-08-02 13:06:59
   ブロックハッシュ: f1d77d06ff06d16...
   トランザクション: 512 件
   Gas使用量      : 29,981,842 / 60,000,000
------------------------------------------------------------
 [NEW BLOCK] 25664703
   タイムスタンプ : 2026-08-02 13:07:11
   ブロックハッシュ: 16a79d21397b73d...
   トランザクション: 288 件
   Gas使用量      : 49,906,836 / 60,000,000
------------------------------------------------------------

 デモ実行（5ループ）が完了しました。
